# Physics Simulation of 3D Gaussian Splats Using Simplicits - Now With Collisions!

Let's simulate 3D Gaussian Splat objects using [Simplicits](https://research.nvidia.com/labs/toronto-ai/simplicits/), fully integrated into the Kaolin Library. We will be able to set up and interactively view the simulation directly in this Jupyter notebook.

With v0.18.0, Kaolin also supports collision handling between objects, which we will also show here.

<img src="../../../assets/physics_bulldozer.gif" alt="image info" width="500px"/>

## Installation and Requirements
For splat rendering, we will be relying on [Nerfstudio Gsplat](https://github.com/nerfstudio-project/gsplat).

We have recently tested this notebook with the following environment. Please follow [Kaolin Installation docs](https://kaolin.readthedocs.io/en/latest/notes/installation.html) to install Kaolin. 
- python 3.11.10
- cuda 12.4
- pytorch 2.5.1
- setuptools 70.1.1

In [ ]:
### Install necessary packages
!pip install -q plyfile k3d matplotlib gsplat

### Import Kaolin Library and Other Requirements

In [ ]:
import copy
import ipywidgets
import json
import kaolin

import matplotlib.pyplot as plt
import numpy as np
import os
import logging
import sys
import time
import threading  
import k3d
from pathlib import Path
from functools import partial
import warp as wp

import torch
import gsplat

from IPython.display import display
from ipywidgets import Button, HBox, VBox

logging.basicConfig(level=logging.INFO, stream=sys.stdout, format="%(asctime)s|%(levelname)8s| %(message)s")
logger = logging.getLogger(__name__)

%load_ext autoreload
%autoreload 2

sys.path.append(str(Path("..")))
from tutorial_common import COMMON_DATA_DIR, FILE_DIR

def log_tensor(t, name, **kwargs):
    """ Debugging util, e.g. call: log_tensor(t, 'my tensor', print_stats=True) """
    logger.info(kaolin.utils.testing.tensor_info(t, name=name, **kwargs)) 

## Download Splat Models from AWS
Lets grab two pre-trained 3D Gaussian Splat models from AWS.
We can unzip and set the splat model path below to the correct `.ply` file.

In [ ]:
# Download and unzip the nerfsynthetic bulldozer
!if test -d output/dozer; then echo "Pretrained bulldozer splats already exist."; else wget https://nvidia-kaolin.s3.us-east-2.amazonaws.com/data/dozer.zip -P output/; unzip output/dozer.zip -d output/; fi;
model_path = 'output/dozer/point_cloud/iteration_30000/point_cloud.ply'

# Download and unzip the doll splat, captured and trained by the Kaolin team (please cite Kaolin if you use this model)
!if [ -e output/BluehairRagdoll.usdc ]; then echo "Pretrained doll splats already exist."; else wget https://nvidia-kaolin.s3.us-east-2.amazonaws.com/data/BluehairRagdoll.usdc -P output/; fi;
model_path2 = 'output/BluehairRagdoll.usdc' 

### Load 3D Gaussian Splat Models

After the setup, we can load and use Kaolin to display the splat model within the Jupyter notebook.

In [ ]:
gaussians = kaolin.io.import_gaussiancloud(model_path).cuda()
gaussians2 = kaolin.io.import_gaussiancloud(model_path2).cuda()

## Interactive Rendering Using Kaolin Visualizer

In order to easily view splats in the notebook, let's set up Gaussian Splat rendering using Kaolin camera conventions.
You should be able to see the rendering below this cell and to control the camera with your left mouse button.

In [ ]:
resolution = 512
default_cam = kaolin.render.camera.Camera.from_args(
        eye=torch.ones((3,)) * 2, at=torch.zeros((3,)), up=torch.tensor([0., 0., 1.]),
        fov=torch.pi * 45 / 180, height=resolution, width=resolution)

class GaussianRenderer:
    """ Define a rendering closure. """
    def __init__(self, gaussians, downscale_factor=1):
        self.gaussians = gaussians
        self.downscale_factor = downscale_factor

    def downscale_camera(self, in_cam):
        lowres_cam = copy.deepcopy(in_cam)
        lowres_cam.width = in_cam.width // self.downscale_factor
        lowres_cam.height = in_cam.height // self.downscale_factor
        return lowres_cam

    def fast_render(self, camera):
        if self.downscale_factor > 1:
            camera = self.downscale_camera(camera)
        return self(camera)

    def __call__(self, camera):
        # Convert kaolin camera to nerfstudio gsplat camera
        cam_params = kaolin.render.camera.kaolin_camera_to_gsplat_nerfstudio(camera.cuda())
        # Render gaussians using the gsplat rendering function
        render_colors, render_alphas, info = gsplat.rendering.rasterization(
            self.gaussians.positions,
            self.gaussians.orientations,
            self.gaussians.scales,
            self.gaussians.opacities,
            self.gaussians.sh_coeff,
            sh_degree=self.gaussians.sh_degree,
            **cam_params
        )
        return  (torch.clamp(render_colors[0], 0, 1) * 255).to(torch.uint8).detach().cpu()
        
static_scene_viz = kaolin.visualize.IpyTurntableVisualizer(
    resolution, resolution, copy.deepcopy(default_cam), GaussianRenderer(gaussians), 
    focus_at=None, world_up_axis=2, max_fps=12, img_quality=75, img_format='JPEG')
static_scene_viz.show() 

## Creating and Training Simplicits Objects from Points
[Simplicits](https://research.nvidia.com/labs/toronto-ai/simplicits/) is a mesh-free, representation-agnostic method for simulating elastic deformations. We can use it to simulate Gaussian Splats at interactive rates within the Jupyter notebook. In order to simulate any point-sampled geometry, such as splats, Simplicits first
_trains_ an object specific weight function representing the reduced degrees of freedom for the object. The physics solver then uses this reduced space to solve for deformations during simulation.

Next, let's use the Simplicits API within Kaolin to create, train and simulate splat objects.

First, let's set some material parameters.

In [ ]:
# Physics material parameters (use approximated values, or look them up online)
# We'll create a few presets that can be used
youngs_modulus_presets = {"softest": 2000, "soft": 21000, "medium": 1e6, "stiff": 1e7}
soft_youngs_modulus = youngs_modulus_presets["soft"]  # we will use this for training
poisson_ratio = 0.45
rhos = 100  # kg/m^3
appx_vol = 3  # m^3

### Sampling Within Splat Volume

Because splats tend to occupy the surface of the object, they provide poor sampling of the object's interior. This can affect the quality of the learned reduced space. To sample within the splat volume, we will use Kaolin's utility `kaolin.ops.gaussians.sample_points_in_volume`.  

In [ ]:
densified_pos = kaolin.ops.gaussians.sample_points_in_volume(
    xyz=gaussians.positions,
    scale=gaussians.scales,
    rotation=gaussians.orientations,
    opacity=gaussians.opacities,
    clip_samples_to_input_bbox=False
)
log_tensor(gaussians.positions, 'original_pos', print_stats=True)
log_tensor(densified_pos, 'densified_pos', print_stats=True)

In [ ]:
def visualize_pts_k3d(densified_pos, pos):
    plot = k3d.plot()
    plot += k3d.points(densified_pos.detach().cpu().numpy(), point_size=0.01, color=0x00ff00)
    plot += k3d.points(pos.detach().cpu().numpy(), point_size=0.02, color=0xff0000)
    plot.display()

visualize_pts_k3d(densified_pos, gaussians.positions)

# Storing physical materials info

All the physical materials informations can be stored in `kaolin.physics.simplicits.PhysicsPoints`.
We can save those in an USD file, and it's used to instantiate a `SimplicitsObject`

In [ ]:
physics_points = kaolin.physics.simplicits.PhysicsPoints(
    pts=densified_pos,
    yms=soft_youngs_modulus,
    prs=poisson_ratio,
    rhos=rhos,
    appx_vol=appx_vol
)

### Training
Next we create a `SimplicitsObject` and train its skinning weight functions using the volume samples, visualized above. The simulator will then use these reduced degrees of freedom to drive the simulation.

Because training take some time we show how to save the object as a [baked representation](https://kaolin.readthedocs.io/en/latest/modules/kaolin.physics.simplicits.html#kaolin.physics.simplicits.SkinnedPhysicsPoints) into an USD file. While the baked object doesn't contain the skinning weight function, it's ready to be simulated in a scene.

Use `ENABLE_SIMPLICITS_CACHING` to reuse the object, set it to `False` to force training.

In [ ]:
# # Whether to save reduced degrees of freedom used by the simulator and load from cache automatically
ENABLE_SIMPLICITS_CACHING = True
cache_dir = os.path.join(FILE_DIR, 'cache', 'gaussians_physics')
os.makedirs(cache_dir, exist_ok=True)
logger.info(f'Caching trained skinned physics points objects in {cache_dir}')

In [ ]:
def export_gaussians_and_physics(file_path,
                                 prim_paths: list[str],
                                 gaussianclouds: list[kaolin.rep.GaussianSplatModel],
                                 physics_points: list[kaolin.physics.simplicits.PhysicsPoints],
                                 skinned_physics_points: list[kaolin.physics.simplicits.SkinnedPhysicsPoints],
                                 init_transforms: list[torch.Tensor] = None,
                                 up_axis='Y'):
    """Save Gaussians objects with their physics material and skinned physics points properties into an USD file"""
    assert len(prim_paths) == len(gaussianclouds) == len(physics_points) == len(skinned_physics_points)
    stage = kaolin.io.usd.create_stage(file_path, up_axis)
    try:
        if init_transforms is None:
            init_transforms = [None] * len(prim_paths)
        for i, gaussiancloud in enumerate(gaussianclouds):
            params = gaussiancloud.as_dict()
            del params['sh_degree']
            prim = kaolin.io.usd.add_gaussiancloud(stage, prim_paths[i], **params, local_to_world=init_transforms[i])
            kaolin.io.usd.add_physics_material(stage, prim, physics_points=physics_points[i])
            kaolin.io.usd.add_skinned_physics(stage, prim, skinned_physics_points=skinned_physics_points[i])

        stage.Save()
    finally:
        del stage

def import_gaussians_and_physics(file_path):
    """Load Gaussians, physics materials and skinned physics points properties from an USD file"""
    output = {}
    gaussianclouds = kaolin.io.usd.import_gaussianclouds(file_path)
    for path, gaussiancloud in gaussianclouds.items():
        # TODO(cfujitsang): support for subsets
        output[path] = {
            'gaussiancloud': gaussiancloud,
            'physics_points': kaolin.io.usd.get_physics_material(file_path, path, material_name='default'),
            'skinned_physics_points': kaolin.io.usd.get_skinned_physics(file_path, path, instance_name='default')
        }
    return output

def train_or_load_baked_simplicits_object(file_path,
                                          gaussianclouds: kaolin.rep.GaussianSplatModel = None,
                                          physics_points: kaolin.physics.simplicits.PhysicsPoints = None):
    if ENABLE_SIMPLICITS_CACHING and os.path.exists(file_path):
        logger.info(f"loading cached object at '{file_path}'")
        return import_gaussians_and_physics(file_path)['/World/Gaussians/obj']
    else:
        assert gaussianclouds is not None and physics_points is not None, \
            f"{file_path} doesn't exist, must provide 'gaussianclouds' and 'physics_points'"
        sim_obj = kaolin.physics.simplicits.SimplicitsObject.create_trained(
            physics_points=physics_points, num_samples=2048, model_layers=10, num_handles=41,
            training_num_steps=20000)
        baked_obj = sim_obj.bake(num_qps=2048, renderable_pts=gaussianclouds.positions)
        export_gaussians_and_physics(file_path, ['/World/Gaussians/obj'], [gaussianclouds], [physics_points], [baked_obj])
        return {
            'gaussiancloud': gaussianclouds,
            'physics_points': physics_points,
            'skinned_physics_points': baked_obj
        }

data = train_or_load_baked_simplicits_object(os.path.join(cache_dir, 'simplicits_truck.usdc'),
                                             gaussians, physics_points)
gaussians = data['gaussiancloud'].cuda()
physics_points = data['physics_points'].cuda()
baked_obj = data['skinned_physics_points'].cuda()

## Setup Simulated Scene Using Simplicits Easy API
Lets create an empty scene with default parameters, then reset the max number of newton steps for faster runtimes.

**Note:** be patient, some of the steps below take time, as we need to build matrices used during simulation.

In [ ]:
scene = kaolin.physics.simplicits.SimplicitsScene() # Create a default scene # default empty scene

scene.max_newton_steps = 3 #Convergence might not be guaranteed at few NM iterations, but runs very fast
scene.timestep = 0.03
scene.newton_hessian_regularizer = 1e-5

Now we add our object to the scene. We use 2048 cubature points to integrate over.

In [ ]:
# The scene copies it into an internal SimulatableObject utility class
obj_idx = scene.add_object(baked_obj)

Lets set set gravity and floor forces on the scene

In [ ]:
# Add gravity to the scene
scene.set_scene_gravity(acc_gravity=torch.tensor([0, 0, 9.8]))
# Add floor to the scene
scene.set_scene_floor(floor_height=-0.7, floor_axis=2, floor_penalty=1000, flip_floor=False)

We can play around with the material parameters of the object, indicated via object_idx

## Simulating and Interactive Visualizing 

That's it! We are ready to simulate. Let's just make sure we can visualize the simulation as it is running.

### Handling Splat Deformation

As the splats deform, we must update their attributes using the transforms predicted by the simulator.
For this, we will need the reduced degrees of freedom and ability to apply linear blend skinning to splats. These utilities can be found in `gaussian_utils.py` relative to this notebook.

In [ ]:
class SingleGaussianObjectDynamicRenderer(GaussianRenderer):
    def __init__(self, gaussians, scene, downscale_factor=1):
        self.rest_gaussians = copy.deepcopy(gaussians)
        self.gaussians = copy.deepcopy(gaussians)
        self.downscale_factor = downscale_factor
        self.scene = scene
        self.reset()

    def run_sim_step(self):
        self.scene.run_sim_step()
        self.update_gaussians()

    def update_gaussians(self):
        with torch.no_grad():
            per_pt_transforms = self.scene.get_object_point_transforms(0, 'rendered')
            self.gaussians = self.rest_gaussians.as_transformed(per_pt_transforms)

    def reset(self):
        self.scene.reset_scene()
        self.update_gaussians()

renderer = SingleGaussianObjectDynamicRenderer(gaussians, scene, 8)

### Threading

We will run simulation in a separate thread, so it is possible to interact with the viewer as the simulation is running (in fact, it's encouraged). We'll reuse these utils for this and the multi-object simulation below.

In [ ]:
global sim_thread
sim_thread = None

def start_simulation(visualizer, num_iterations):    
    global sim_thread

    def sim_function():
        for s in range(num_sim_iterations):
            with visualizer.out:
                visualizer.render.run_sim_step()
                print(".", end="")
            visualizer.render_update()

    with visualizer.out:
        if sim_thread is not None:
            sim_thread.join()
        sim_thread = threading.Thread(target=sim_function, daemon=True)
        sim_thread.start()

def reset_simulation(reset_function, visualizer):
    with visualizer.out:
        reset_function()
    visualizer.render_update()

### Simulation: Let's bring everything together!

In [ ]:
num_sim_iterations = 100

resolution = 512
sim_visualizer = kaolin.visualize.IpyTurntableVisualizer(
    resolution, resolution, copy.deepcopy(default_cam),
    renderer, fast_render=renderer.fast_render,
    focus_at=torch.tensor([0, 0, 0.0]),
    world_up_axis=2, max_fps=12, img_quality=75, img_format='JPEG')

buttons = [Button(description=x) for x in
           ['Run Sim', 'Reset']]
buttons[0].on_click(lambda e: start_simulation(sim_visualizer, num_sim_iterations))
buttons[1].on_click(lambda e: reset_simulation(renderer.reset, sim_visualizer))

sim_visualizer.render_update()
display(VBox([HBox([sim_visualizer.canvas, VBox(buttons)]), sim_visualizer.out]))

# Part 2: Multiple Objects and Collisions

It's time to make this simulation more exciting. Let's train and add the second object that we loaded above to the simulation.

### Train Second Simplicits Object

As before, we will sample and visualizer points in the object volume. Then, we'll train and cache a simplicits object.

In [ ]:
densified_pos2 = kaolin.ops.gaussians.sample_points_in_volume(
    xyz=gaussians2.positions.detach(), 
    scale=gaussians2.scales.detach(),
    rotation=gaussians2.orientations.detach(),
    opacity=gaussians2.opacities.detach(),
    clip_samples_to_input_bbox=False
)
log_tensor(gaussians2.positions, 'original_pos', print_stats=True)
log_tensor(densified_pos2, 'densified_pos', print_stats=True)

In [ ]:
visualize_pts_k3d(densified_pos2, gaussians2.positions)

In [ ]:
physics_points2 = kaolin.physics.simplicits.PhysicsPoints(
    pts=densified_pos2,
    yms=soft_youngs_modulus,
    prs=poisson_ratio,
    rhos=rhos,
    appx_vol=appx_vol
)

In [ ]:
data = train_or_load_baked_simplicits_object(os.path.join(cache_dir, 'simplicits_doll.usdc'),
                                             gaussians2, physics_points2)
gaussians2 = data['gaussiancloud'].cuda()
physics_points2 = data['physics_points'].cuda()
baked_obj2 = data['skinned_physics_points'].cuda()

### Set Up New Scene

We'll set up a new scene to make sure the previous simulation cell is still functional.

In [ ]:
init_transform = None
init_transform2 = torch.tensor([[1,0,0,0],
                                [0,1,0,0],
                                [0,0,1,1], 
                                [0,0,0,1]], dtype=torch.float32, 
                               device=gaussians2.positions.device)

In [ ]:
scene2 = kaolin.physics.simplicits.SimplicitsScene() # Create a default scene # default empty scene

scene2.max_newton_steps = 3 #Convergence might not be guaranteed at few NM iterations, but runs very fast
scene2.timestep = 0.03
scene2.newton_hessian_regularizer = 1e-5

We'll add 2 objects this time, offsetting the doll in the z direction.

In [ ]:
scene2_obj_idx = scene2.add_object(baked_obj, init_transform=init_transform)
scene2_obj_idx2 = scene2.add_object(baked_obj2, init_transform=init_transform2) 

We'll set up forces as before.

In [ ]:
# Add gravity to the scene
scene2.set_scene_gravity(acc_gravity=torch.tensor([0, 0, 9.8]))
# Add floor to the scene
scene2.set_scene_floor(floor_height=-0.7, floor_axis=2, floor_penalty=1000, flip_floor=False)

### Enable Collisions (new!)

We will enable inter-object collisions here. 

In [ ]:
scene2.enable_collisions(
    collision_particle_radius=0.1, # radius of each collision particle - energy starts accumulating at r
    detection_ratio=1.5, # radius * detection ratio is the area that is searched for potential contact
    impenetrable_barrier_ratio=0.25, # radius * barrier is the distance at which energy is infinite
    collision_penalty=1000.0, # coefficient of collision energy, force, gradient
    max_contact_pairs=10000, # the maximum number of particle contact pairs to allow
    friction=0.5, # friction coefficient
)

### Handle Deforming and Rendering Multiple Gaussians

Because the renderer is not set up to render multi-object scenes, we need to do a little work in order to visualize the simulation. Let's concatenate both objects into a single GaussianSplatModel. The `SimplicitsScene` will provide per-object transform that we can concatenate to transform the whole scene. (We could also transform each object independently and concatenate them.

In [ ]:
combined_gaussians = kaolin.rep.GaussianSplatModel.cat([gaussians, gaussians2])

class MultiGaussianObjectDynamicRenderer(SingleGaussianObjectDynamicRenderer):
    def update_gaussians(self):
        with torch.no_grad():
            per_pt_transforms = []
            for k in sorted(self.scene.sim_obj_dict.keys()):
                per_pt_transforms.append(self.scene.get_object_point_transforms(k, 'rendered'))
            per_pt_transforms = torch.cat(per_pt_transforms, dim=0)
            self.gaussians = self.rest_gaussians.as_transformed(per_pt_transforms)

multi_renderer = MultiGaussianObjectDynamicRenderer(combined_gaussians, scene2, 8)

Let's make sure we can deform both objects using the learned degrees of freedom, which the Simplicits simulator is using to predict deformations.

### Simulate and Visualize

Now we are ready to run the simulation and visualize it.

In [ ]:
num_sim_iterations = 100

resolution = 512
multi_sim_visualizer = kaolin.visualize.IpyTurntableVisualizer(
    resolution, resolution, copy.deepcopy(default_cam),
    multi_renderer, fast_render=multi_renderer.fast_render,
    focus_at=torch.tensor([0, 0, 0.0]),
    world_up_axis=2, max_fps=12, img_quality=75, img_format='JPEG')

buttons = [Button(description=x) for x in
           ['Run Sim', 'Reset']]
buttons[0].on_click(lambda e: start_simulation(multi_sim_visualizer, num_sim_iterations))
buttons[1].on_click(lambda e: reset_simulation(multi_renderer.reset, sim_visualizer))

multi_sim_visualizer.render_update()
display(VBox([HBox([multi_sim_visualizer.canvas, VBox(buttons)]), multi_sim_visualizer.out]))

# Saving / Loading the whole scene
You can save and load a whole simulatable scene in USD (after loading, restart from `scene2 = kaolin.physics.simplicits.SimplicitsScene() ...`)

In [ ]:
multi_renderer.reset()
export_gaussians_and_physics(os.path.join(cache_dir, 'whole_scene.usdc'),
                             prim_paths=['/World/Gaussians/obj', '/World/Gaussians/obj2'],
                             gaussianclouds=[gaussians, gaussians2],
                             init_transforms=[init_transform, init_transform2],
                             physics_points=[physics_points, physics_points2],
                             skinned_physics_points=[baked_obj, baked_obj2])

In [ ]:
scene_gaussians_and_physics = import_gaussians_and_physics(os.path.join(cache_dir, "whole_scene.usdc"))
gaussians = scene_gaussians_and_physics['/World/Gaussians/obj']['gaussiancloud'].cuda()
gaussians2 = scene_gaussians_and_physics['/World/Gaussians/obj2']['gaussiancloud'].cuda()
physics_points = scene_gaussians_and_physics['/World/Gaussians/obj']['physics_points'].cuda()
physics_points2 = scene_gaussians_and_physics['/World/Gaussians/obj2']['physics_points'].cuda()
baked_obj = scene_gaussians_and_physics['/World/Gaussians/obj']['skinned_physics_points'].cuda()
baked_obj2 = scene_gaussians_and_physics['/World/Gaussians/obj2']['skinned_physics_points'].cuda()
init_transform = gaussians.transform
init_transform2 = gaussians2.transform
gaussians.transform = None
gaussians2.transform = None